# 🔍 Data Validation Example

This notebook demonstrates how to validate data and handle errors using `quarantine` mode.

## Scenario
We check for:
1. **Unique Email**: Duplicate emails are quarantined.
2. **Valid Age**: Age must be between 0 and 120. Outliers are quarantined.
3. **User Type**: Must not be null.

## 🔧 Setup

Install `mlprep-rust` package from PyPI.

In [ ]:
!pip install mlprep-rust -q

## 📁 Generate Dirty Data

Generate sample data with intentional issues:
- Duplicate emails
- Invalid age values
- Null user_type values

In [ ]:
import pandas as pd
import numpy as np

def generate_dirty_data():
    np.random.seed(42)
    n_rows = 100
    
    data = {
        'id': range(1, n_rows + 1),
        'email': [f'user{i}@example.com' for i in range(n_rows)],
        'age': np.random.randint(15, 60, size=n_rows),
        'user_type': np.random.choice(['admin', 'user', 'guest'], size=n_rows)
    }
    
    df = pd.DataFrame(data)
    
    # Introduce bad data
    # Duplicate emails
    df.loc[0, 'email'] = 'duplicate@example.com'
    df.loc[1, 'email'] = 'duplicate@example.com'
    
    # Invalid age
    df.loc[2, 'age'] = -5
    df.loc[3, 'age'] = 150
    
    # Null user_type
    df.loc[4, 'user_type'] = np.nan
    
    df.to_csv('dirty_data.csv', index=False)
    print("Generated dirty_data.csv with 100 rows (including intentional errors)")
    return df

df = generate_dirty_data()
print("\n🚨 Intentional errors in data:")
print("\nRows 0-1 (duplicate emails):")
print(df.iloc[:2][['id', 'email']])
print("\nRows 2-3 (invalid ages):")
print(df.iloc[2:4][['id', 'age']])
print("\nRow 4 (null user_type):")
print(df.iloc[4:5][['id', 'user_type']])

## 📝 Create Pipeline Configuration

Create the `pipeline.yaml` file with validation rules and quarantine mode.

In [ ]:
pipeline_yaml = """
name: data_validation
inputs:
  - path: dirty_data.csv
    format: csv

steps:
  - type: validate
    checks:
      columns:
        - name: email
          unique: true
        - name: age
          range: [0, 120]
        - name: user_type
          not_null: true
    mode: quarantine

outputs:
  - path: clean_output.parquet
    format: parquet
"""

with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml.strip())

print("Created pipeline.yaml")
print(pipeline_yaml)

## 🚀 Run Pipeline

Execute the pipeline with validation.

In [ ]:
!mlprep run pipeline.yaml

## ✅ Verify Output

Check the clean output and quarantine folder.

In [ ]:
import os
import pandas as pd

# Load the clean output
if os.path.exists('clean_output.parquet'):
    clean_df = pd.read_parquet('clean_output.parquet')
    print(f"✅ Clean output: {len(clean_df)} rows")
    print("\nSample of clean data:")
    print(clean_df.head(10))
else:
    print("❌ clean_output.parquet not found")

# Check quarantine folder
print("\n" + "="*50)
print("Checking quarantine folder...")
if os.path.exists('quarantine'):
    for f in os.listdir('quarantine'):
        print(f"\n📂 Found: quarantine/{f}")
        quarantine_df = pd.read_csv(f'quarantine/{f}')
        print(f"Quarantined rows: {len(quarantine_df)}")
        print(quarantine_df)
else:
    print("⚠️ No quarantine folder found")

## 📊 Summary

Compare input and output data quality.

In [ ]:
input_df = pd.read_csv('dirty_data.csv')

print("📊 Data Quality Summary")
print("="*50)
print(f"Input rows: {len(input_df)}")
if os.path.exists('clean_output.parquet'):
    clean_df = pd.read_parquet('clean_output.parquet')
    print(f"Clean output rows: {len(clean_df)}")
    print(f"Rows quarantined: {len(input_df) - len(clean_df)}")
    
    # Validate clean data
    print("\n✅ Validation Results:")
    print(f"  - All emails unique: {clean_df['email'].is_unique}")
    print(f"  - All ages in [0, 120]: {((clean_df['age'] >= 0) & (clean_df['age'] <= 120)).all()}")
    print(f"  - No null user_type: {clean_df['user_type'].notna().all()}")